In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!pip install transformers datasets scikit-learn openpyxl -q

In [ ]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)

from sklearn.metrics import roc_auc_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
train = pd.read_excel("toxic_labeled.xlsx")
test  = pd.read_excel("toxic_no_label_evaluation.xlsx")

print("Train shape:", train.shape)
print("Label distribution:\n", train['label'].value_counts())

In [ ]:
# Drop null text rows
train = train.dropna(subset=['text']).reset_index(drop=True)
test['text'] = test['text'].fillna('')

print("After cleaning — Train shape:", train.shape)

TEXT_COL  = 'text'
LABEL_COL = 'label'
y = train[LABEL_COL].values

In [ ]:
vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2), analyzer='char_wb')
X = vec.fit_transform(train[TEXT_COL])

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
scores = cross_val_score(lr, X, y, cv=5, scoring='roc_auc')
print(f"Baseline CV ROC-AUC: {scores.mean():.4f}")

# Save baseline
lr.fit(X, y)
baseline_preds = (lr.predict_proba(vec.transform(test[TEXT_COL]))[:, 1] >= 0.5).astype(int)
test_baseline = pd.read_excel("toxic_no_label_evaluation.xlsx")
test_baseline['label'] = baseline_preds
test_baseline.to_excel("toxic_no_label_baseline.xlsx", index=False)
print("✅ Baseline saved")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

MODEL = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True,
                     padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(
    train[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: 'label'})
)
train_ds = train_ds.map(tokenize, batched=True)
split = train_ds.train_test_split(test_size=0.1, seed=42)
print("✅ Dataset ready")
print("Train size:", len(split['train']))
print("Val size:", len(split['test']))

In [ ]:
#  Model Training

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="./c3_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    seed=42,
    logging_steps=50,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
)
trainer.train()

In [ ]:
# ── 8. Evaluate ROC-AUC (SCREENSHOT THIS) ────────────────
val_preds = trainer.predict(split['test'])
val_proba = torch.softmax(
    torch.tensor(val_preds.predictions), dim=1
).numpy()[:, 1]
val_true = np.array(split['test']['label'])

roc_auc = roc_auc_score(val_true, val_proba)
val_labels_pred = (val_proba >= 0.5).astype(int)

print("\n" + "="*50)
print("VALIDATION RESULTS —")
print("="*50)
print(classification_report(val_true, val_labels_pred,
      target_names=['Non-Toxic (0)', 'Toxic (1)']))
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("="*50)

In [ ]:
# ── 9. Predict & Save Final Submission ───────────────────
test_ds = Dataset.from_pandas(test[[TEXT_COL]])
test_ds = test_ds.map(tokenize, batched=True)

final_preds = trainer.predict(test_ds)
final_proba = torch.softmax(
    torch.tensor(final_preds.predictions), dim=1
).numpy()[:, 1]
final_labels = (final_proba >= 0.5).astype(int)

# Fill label column
test_final = pd.read_excel("toxic_no_label_evaluation.xlsx")
test_final['label'] = final_labels

# Verify
print("Predicted label values:", pd.Series(final_labels).value_counts().to_dict())
print("Expected: 0 / 1")
print("Null labels:", test_final['label'].isnull().sum())
print("Total rows:", len(test_final))

test_final.to_excel("toxic_no_label_evaluation.xlsx", index=False)
print("✅ Final submission saved — toxic_no_label_evaluation.xlsx")

In [ ]:
from google.colab import files
files.download("toxic_no_label_evaluation.xlsx")